In [ ]:
# Structure comparison: pLDDT vs. TM-score (ESM2)

# Compares the structures generated by the ESM2 pipeline (ESM2 followed by ColabFold) with the reference toxins 4K09 and 1KBA.

# The notebook reads the filtered TM-score CSV file and calculates the mean pLDDT of each filtered ColabFold model from the B-factor column of its CA atoms. 
# The mean pLDDT is then plotted against the TM-score (`tm1`):
# - scatter plots for the reference toxins 4K09 and 1KBA and for all data (displayed only, not saved),
# - KDE density plot of all data (saved as PDF in `PLOTS_DIR`).

# The structural similarity (StS) threshold is drawn at a TM-score of 0.6.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit these values to match your setup.
# ---------------------------------------------------------------------------

# Directory containing the TM-score CSV file.
BASE_DIR = "/path_to_your_base_directory/ESM2_filtered"

# Directory holding the filtered PDB files.
FILTERED_PDB_DIR = "/path_to_your_base_directory/ESM2_filtered/4K09_fixed_AS"

# Directory where the generated plots (PDFs) are written.
PLOTS_DIR = "/path_to_your_base_directory/ESM2"

# Model label used in the plot legends (CF = ColabFold).
MODEL_LABEL = "ESM2 + CF"

# Input TM-score CSV filename.
TM_ALPHAFOLD_FILENAME = "TM_score_4K09_fixed_AS_filtered.csv"

# Output plot filename.
DENSITY_FILENAME = "density_plddt_vs_tm_4K09_fixed_AS.pdf"
# ---------------------------------------------------------------------------


In [ ]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
def get_mean_plddt(pdb_path):
    """Extract mean pLDDT score (stored in B-factor column) for CA atoms from a PDB file."""
    plddt_scores = []
    if not os.path.exists(pdb_path):
        return None
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                if line[12:16].strip() == "CA":
                    plddt_scores.append(float(line[60:66].strip()))
    return sum(plddt_scores) / len(plddt_scores) if plddt_scores else None

In [ ]:
csvpath_alphafold = os.path.join(BASE_DIR, TM_ALPHAFOLD_FILENAME)

In [ ]:
df_alphafold = pd.read_csv(csvpath_alphafold, low_memory=False)
df_alphafold['processing_status'] = MODEL_LABEL

In [ ]:
filtered_pdb_dir = Path(FILTERED_PDB_DIR)
print("Scanning folder structure for PDB files...")
pdb_path_map = {file.name: file for file in filtered_pdb_dir.rglob("*.pdb")}
print(f"{len(pdb_path_map)} PDB files found.")

def get_plddt_fast(fname):
    pdb_path = pdb_path_map.get(fname)
    return get_mean_plddt(pdb_path) if pdb_path else None

df_alphafold['plddt'] = df_alphafold['generated_file'].apply(get_plddt_fast)

In [ ]:
print("First rows of the DataFrame:")
print(df_alphafold.head())
print("\nCount per processing status:")
print(df_alphafold['processing_status'].value_counts())

In [ ]:
# Global plot styling
plt.rcParams['axes.titlesize'] = 24
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['legend.title_fontsize'] = 18

In [ ]:
df_alphafold_4k09 = df_alphafold[df_alphafold['reference_file'] == 'pdb4k09.ent']

In [ ]:
# Scatter plot: pLDDT vs TM-score (4K09 reference)
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_alphafold_4k09, x='plddt', y='tm1', color='tab:blue', label=f'{MODEL_LABEL} 4K09.pdb', alpha=0.8, s=40)
plt.title('Protein structural similarity (StS) comparison')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(title='Processing status', loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

In [ ]:
df_alphafold_1kba = df_alphafold[df_alphafold['reference_file'] == 'pdb1kba.ent']

In [ ]:
# Scatter plot: pLDDT vs TM-score (1KBA reference)
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_alphafold_1kba, x='plddt', y='tm1', color='tab:blue', label=f'{MODEL_LABEL} 1KBA.pdb', alpha=0.8, s=40)
plt.title('Protein structural similarity (StS) comparison')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(title='Processing status', loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

In [ ]:
# Scatter plot: all data
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_alphafold, x='plddt', y='tm1', color='tab:blue', label=MODEL_LABEL, alpha=0.8, s=40)
plt.title('Protein structural similarity (StS) comparison')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(title='Processing status', loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

In [ ]:
# KDE density plot: pLDDT vs TM-score (all data)
plt.figure(figsize=(10, 7))
sns.kdeplot(data=df_alphafold, x='plddt', y='tm1', fill=True, cmap='managua',
            thresh=0.02, levels=20, clip=((69, 100), (0.0, 1.0)), cbar=True,
            cbar_kws={'label': 'density'})
plt.title('Structural similarity (StS) density plot')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, DENSITY_FILENAME), format="pdf")
plt.show()